# Step 0 - Download the raw Lichess file, check it and profile it

**What this notebook does (in order)**
1. **Download** one pinned monthly Lichess file (resumable: if the connection drops, just run the cell again).
2. **Verify and record** it: exact byte size, SHA-256, and a `download_manifest.json`.
3. **Profile** the whole file in one streaming pass (no sampling): what is in it, before any filtering.
4. **Check the data quality** in the same pass: duplicate game IDs (all games), result / date / Elo consistency (all games), eval completeness (all analysed games),
   legality of the moves and game length (every `LEGALITY_EVERY`-th game, replayed with python-chess), plausibility of the eval values (every `EVAL_EVERY`-th analysed game).
5. **Plot** everything (PNG for the thesis), save every table as CSV, and cross-check the key numbers against the earlier independent run.

**Reproducible because:** the month, URL, size, SHA-256, package versions, a fingerprint of the profiling code and a hash of all result tables are written to the manifests.
Same file + same notebook = the same `results_sha256` (also on another operating system: tables are written with fixed line endings and number formats).

**Time and space:** a recent month is roughly 30 GB to download; the profiling pass takes roughly 1 to 1.5 hours; the duplicate check needs a temporary 0.8 GB file and about 1 GB of free RAM at the end.
The pass saves checkpoints, so you can stop it (Ctrl+C / close VS Code) and run the same cell again to continue.
Try `MAX_GAMES = 1_000_000` first for a quick test (it writes to its own folder `<month>_test1000000`).

Needs (once): `pip install -r requirements.txt`. Open the notebook from the project folder.

In [ ]:
import io, os, re, sys, json, time, pickle, hashlib, platform, datetime, inspect
from array import array
from collections import Counter
from importlib import metadata
from pathlib import Path
import numpy as np
import pandas as pd
import requests, zstandard
import chess, chess.pgn
import matplotlib.pyplot as plt

# ===================== PARAMETERS - the only place to edit =====================
import importlib, config; importlib.reload(config)   # settings live in config.py (restart the kernel after editing it)
from config import MONTH, BASE_URL, MIN_EST_SECONDS, MAX_RATING_GAP   # pinned month/URL and the funnel thresholds: defined once, in config.py
MAX_GAMES = None          # None = whole file. Use e.g. 1_000_000 for a quick test run (own output folder)
TAG_SAMPLE_GAMES = 2_000_000   # header-tag fill rates are counted on the first N games only (the tag set is stable)
CHECKPOINT_EVERY = 5_000_000   # games between checkpoints
RECHECK_HASH = True       # True: always recompute the file's SHA-256 (1-3 min). False: reuse the saved one if the size is unchanged
RAW_OVERRIDE = None       # path to an existing local file (skips the download); None = normal
CHECK_DUPLICATE_IDS = True     # every game's ID goes to a temporary 8-bytes-per-game file; checked for repeats at the end
LEGALITY_EVERY = 20_000        # replay every N-th game with python-chess: legal moves, result, length (about 4,800 games per month)
EVAL_EVERY = 10                # check the eval VALUES of every N-th game that carries evals (about 0.9 million games)
# ===============================================================================

FILE = f"lichess_db_standard_rated_{MONTH}.pgn.zst"
URL = BASE_URL + FILE
PROJECT = Path.cwd()
RAW = Path(RAW_OVERRIDE) if RAW_OVERRIDE else PROJECT / "data" / "raw" / FILE
RUN = MONTH if MAX_GAMES is None else f"{MONTH}_test{MAX_GAMES}"      # test runs never mix with the real run
OUT = PROJECT / "outputs" / "step0_dataset" / RUN
TABLES, FIGS = OUT / "tables", OUT / "figures"
for d in (RAW.parent, TABLES, FIGS):
    d.mkdir(parents=True, exist_ok=True)
CODE_VERSION = "step0-v3"      # the profiling code is also fingerprinted automatically (see section 3)
print("month:", MONTH, "| file:", FILE, "\nraw file path:", RAW, "\noutputs:", OUT)

## 1. Download (resumable)
Streams to `<file>.part` and renames it when the size matches what the server announced.
If anything goes wrong, **run this cell again** and it continues from where it stopped.

In [ ]:
def remote_size(url):
    r = requests.head(url, allow_redirects=True, timeout=60); r.raise_for_status()
    return int(r.headers["Content-Length"])

def download(url, dest, expected, chunk=1 << 22, tick=15):
    part = dest.with_name(dest.name + ".part")
    if dest.exists():                                   # an incomplete file with the final name: continue it
        if dest.stat().st_size > expected:
            raise RuntimeError(f"{dest} is larger than the server file; delete it and run again")
        dest.replace(part)
    have = part.stat().st_size if part.exists() else 0
    if have > expected:
        part.unlink(); have = 0
    if have < expected:
        headers = {"Range": f"bytes={have}-"} if have else {}
        with requests.get(url, headers=headers, stream=True, timeout=60) as r:
            r.raise_for_status()
            mode = "ab" if (have and r.status_code == 206) else "wb"
            done = have if mode == "ab" else 0
            print(f"{'Resuming at' if mode == 'ab' else 'Starting at'} {done/1e9:.2f} GB of {expected/1e9:.2f} GB")
            t0 = last = time.time(); start = done
            with open(part, mode) as f:
                for blk in r.iter_content(chunk):
                    f.write(blk); done += len(blk)
                    if time.time() - last > tick:
                        last = time.time(); rate = (done - start) / (last - t0) / 1e6
                        print(f"  {done/expected:6.1%}  {done/1e9:6.2f} GB  {rate:5.1f} MB/s", flush=True)
    if part.stat().st_size != expected:
        raise RuntimeError(f"incomplete: {part.stat().st_size:,} of {expected:,} bytes. Run this cell again to resume.")
    part.replace(dest)

if RAW_OVERRIDE:
    expected = RAW.stat().st_size
    print("Using local file (no download):", RAW)
else:
    expected = remote_size(URL)
    if RAW.exists() and RAW.stat().st_size == expected:
        print(f"Already downloaded and complete: {RAW.name} ({expected/1e9:.2f} GB)")
    else:
        download(URL, RAW, expected)
        print("Download complete:", RAW)

## 2. Verify and record (reproducibility anchor)
Records the exact size and SHA-256 of the file. If Lichess publishes a checksum list we compare against it (best effort; not every server provides one).

In [ ]:
def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

def pkg_versions():
    out = {}
    for p in ["requests", "zstandard", "numpy", "pandas", "matplotlib", "chess"]:
        try: out[p] = metadata.version(p)
        except metadata.PackageNotFoundError: pass
    return out

MAN = OUT / "download_manifest.json"
size = RAW.stat().st_size
old = json.loads(MAN.read_text()) if MAN.exists() else {}
if old.get("bytes") == size and old.get("sha256") and not RECHECK_HASH:
    sha = old["sha256"]; print("Reusing saved SHA-256 (size unchanged).")
else:
    print("Computing SHA-256 (about 1-3 minutes for 30 GB)...")
    sha = sha256_file(RAW)

official = None
try:
    r = requests.get(BASE_URL + "sha256sums.txt", timeout=30)
    if r.ok:
        for line in r.text.splitlines():
            parts = line.split()
            if len(parts) >= 2 and parts[-1].lstrip("*") == FILE:
                official = parts[0]
except requests.RequestException:
    pass

manifest = dict(step="step0_download", month=MONTH, file=FILE, url=URL, bytes=size, sha256=sha,
                official_sha256=official, matches_official=(None if official is None else official == sha),
                timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                python=sys.version.split()[0], platform=platform.platform(), packages=pkg_versions())
MAN.write_text(json.dumps(manifest, indent=2))
print(f"{FILE}\n  size   : {size:,} bytes ({size/1e9:.2f} GB)\n  sha256 : {sha}")
print("  official checksum:", "not available from the server" if official is None else ("MATCH" if official == sha else "MISMATCH - re-download!"))

## 3. Profile and check the whole file (one streaming pass, resumable)
For every game we read the header tags and the move text and update counters. Nothing is sampled except (a) the header-tag fill rates (first `TAG_SAMPLE_GAMES` games),
(b) the move-legality replay (every `LEGALITY_EVERY`-th game, a fixed rule, so it is reproducible) and (c) the eval-value checks (every `EVAL_EVERY`-th analysed game).
A full legality replay of 95 million games would take days; the later pipeline steps replay every selected game anyway.
Stop any time and rerun this cell: it resumes from the last checkpoint and gives the identical result.
If you change the code in this cell, the checkpoint is ignored automatically (the code is fingerprinted).

In [ ]:
THRESH = [1800, 2000, 2200, 2400, 2500, 2600, 2700]
WANT = {b"WhiteElo", b"BlackElo", b"TimeControl", b"Termination", b"Result", b"WhiteTitle", b"BlackTitle", b"ECO", b"UTCDate", b"Event", b"Site"}
SPEED_KEYS = (b"UltraBullet", b"Bullet", b"Blitz", b"Rapid", b"Classical", b"Correspondence")
MOVE_NO = re.compile(rb"(?<![\d.])(\d+)\.(?!\d)")      # '12.' or '12...' but not the '0.' in '[%eval 0.17]'
TC = re.compile(rb"(\d+)\+(\d+)")
EVAL = re.compile(rb"\[%eval ([^\]\s]+)\]")
VALID_RESULTS = (b"1-0", b"0-1", b"1/2-1/2")
MONTH_DOTTED = MONTH.replace("-", ".").encode()          # '2026.10' as in the UTCDate tag
BAND_W, BAND_B, BAND_E = "white_ahead(>=+3.00)", "black_ahead(<=-3.00)", "balanced"

def speed_of(event):
    for k in SPEED_KEYS:
        if k in event: return k.decode()
    return "Other"

def est_seconds(tc):
    m = TC.fullmatch(tc); return int(m[1]) + 40 * int(m[2]) if m else None

def to_elo(b):
    try:
        v = int(b); return v if 0 < v < 4000 else None
    except (TypeError, ValueError):
        return None

def full_moves(mt):
    m = MOVE_NO.findall(mt[-400:]); return int(m[-1]) if m else 0

def new_state():
    keys = ["tags", "result", "term", "speed", "tc", "w", "b", "mn", "gap", "titles", "bot", "eco", "day", "moves",
            "moves_speed", "eval_speed", "clk_speed", "eval_band", "cohort", "quality", "eval_hist", "eval_final", "eval_flags", "legal"]
    S = {k: Counter() for k in keys}; S["n"] = 0; S["eval_games"] = 0; S["dup"] = None; return S

def eval_check(mt, res, S):
    """Eval VALUES of one analysed game: parseable? plausible range? does the last evaluation agree with the result?"""
    F = S["eval_flags"]; F["games_checked"] += 1; last = None
    for v in EVAL.findall(mt):
        F["values"] += 1
        if v[:1] == b"#":
            F["mate_tokens"] += 1; last = -100000 if v[:2] == b"#-" else 100000; continue
        try: cp = int(round(float(v) * 100))
        except ValueError: F["unparsable"] += 1; continue
        if abs(cp) > 3000: F["abs_over_30_pawns"] += 1
        S["eval_hist"][max(-1000, min(1000, cp)) // 100 * 100] += 1
        last = cp
    if last is not None:
        S["eval_final"][(BAND_W if last >= 300 else BAND_B if last <= -300 else BAND_E, res)] += 1

def legality_check(mt, mv, S):
    """Replay one game with python-chess: every move legal? is the length estimate right?"""
    L = S["legal"]; L["games_replayed"] += 1
    try:
        g = chess.pgn.read_game(io.StringIO(mt.decode("utf-8", "replace")))
    except Exception:
        L["parser_exception"] += 1; return
    if g is None: L["no_game_found"] += 1; return
    if g.errors: L["illegal_or_unparsable_moves"] += 1
    plies = sum(1 for _ in g.mainline_moves())
    L["plies_replayed"] += plies
    if (plies + 1) // 2 != mv: L["length_estimate_differs"] += 1

def process(h, mt, S, ids):
    S["n"] += 1; n = S["n"]; Q = S["quality"]
    res = h.get(b"Result", b"?"); S["result"][res] += 1
    S["term"][h.get(b"Termination", b"?")] += 1
    speed = speed_of(h.get(b"Event", b"")); S["speed"][speed] += 1
    tc = h.get(b"TimeControl", b"?"); S["tc"][tc] += 1
    est = est_seconds(tc)
    if est is None: Q["no_clock_time_control"] += 1
    site = h.get(b"Site", b"")
    if len(site) < 8: Q["bad_site_tag"] += 1
    if ids is not None: ids.append(int.from_bytes(site[-8:], "big") if len(site) >= 8 else 0)
    if h.get(b"UTCDate", b"")[:7] != MONTH_DOTTED: Q["utcdate_outside_month"] += 1
    if mt.rstrip().rsplit(b" ", 1)[-1] != res: Q["movetext_result_differs_from_header"] += 1
    w, b = to_elo(h.get(b"WhiteElo")), to_elo(h.get(b"BlackElo"))
    wt, bt = h.get(b"WhiteTitle"), h.get(b"BlackTitle")
    if wt: S["titles"][wt] += 1
    if bt: S["titles"][bt] += 1
    bot = wt == b"BOT" or bt == b"BOT"
    S["bot"][(speed, int(bot))] += 1
    S["eco"][h.get(b"ECO", b"?")] += 1
    S["day"][h.get(b"UTCDate", b"?")] += 1
    has_eval, has_clk = b"[%eval" in mt, b"[%clk" in mt
    S["eval_speed"][(speed, int(has_eval))] += 1; S["clk_speed"][(speed, int(has_clk))] += 1
    mv = full_moves(mt); S["moves"][min(mv, 300)] += 1; S["moves_speed"][(speed, min(mv, 300))] += 1
    if mv == 0: Q["no_moves"] += 1
    if res == b"*": Q["unfinished_result_star"] += 1
    if has_eval:
        S["eval_games"] += 1
        if mt.count(b"[%eval") < 2 * mv - 1: Q["eval_games_with_missing_evals"] += 1
        if S["eval_games"] % EVAL_EVERY == 0: eval_check(mt, res, S)
    if n % LEGALITY_EVERY == 0: legality_check(mt, mv, S)
    if w is None or b is None:
        Q["missing_or_invalid_elo"] += 1; S["eval_band"][(-1, int(has_eval))] += 1
        return
    mn, gap = min(w, b), abs(w - b)
    S["w"][w] += 1; S["b"][b] += 1; S["mn"][mn] += 1; S["gap"][gap] += 1
    S["eval_band"][(mn // 100 * 100, int(has_eval))] += 1
    if mn >= THRESH[0]:
        tier = max(i for i, t in enumerate(THRESH) if mn >= t)
        S["cohort"][(tier, int(gap <= MAX_RATING_GAP), int(est is not None and est >= MIN_EST_SECONDS),
                     int(h.get(b"Termination") == b"Normal"), int(res in VALID_RESULTS), int(not bot), int(has_eval))] += 1

def open_pgn(path):
    raw = open(path, "rb")
    if path.suffix == ".zst":
        return raw, io.BufferedReader(zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(raw), buffer_size=1 << 22)
    return raw, io.BufferedReader(raw, buffer_size=1 << 22)          # plain .pgn (only used for tests)

def make_fp(path, max_games):
    return dict(month=MONTH, size=path.stat().st_size, max_games=max_games, tag_sample=TAG_SAMPLE_GAMES, dup=CHECK_DUPLICATE_IDS,
                legality_every=LEGALITY_EVERY, eval_every=EVAL_EVERY, code=CODE_FP)

def profile(path, max_games, ckpt_path, ids_path):
    fp = make_fp(path, max_games)
    S, skip = new_state(), 0
    if ckpt_path.exists():
        ck = pickle.load(open(ckpt_path, "rb"))
        ids_ok = (not CHECK_DUPLICATE_IDS) or (ids_path.exists() and ids_path.stat().st_size >= ck["state"]["n"] * 8)
        if ck["fingerprint"] == fp and ids_ok:
            S, skip = ck["state"], ck["state"]["n"]; print(f"Resuming from checkpoint: {skip:,} games already done")
        else:
            print("Checkpoint belongs to different settings or code (or its game-ID file is missing) - starting fresh")
    ids_f = None
    if CHECK_DUPLICATE_IDS:
        if skip: ids_f = open(ids_path, "r+b"); ids_f.truncate(skip * 8); ids_f.seek(skip * 8)
        else: ids_f = open(ids_path, "wb")
    ids = array("Q") if ids_f else None
    def flush_ids():
        if ids_f is not None and len(ids): ids.tofile(ids_f); ids_f.flush(); del ids[:]
    total_bytes = path.stat().st_size; t0 = last = time.time(); partial = False
    raw, f = open_pgn(path)
    try:
        hdr, seen = {}, 0
        for line in f:
            if line[:1] == b"[":
                if seen >= skip:
                    sp = line.find(b" "); tag = line[1:sp]
                    if S["n"] < TAG_SAMPLE_GAMES: S["tags"][tag] += 1
                    if tag in WANT: hdr[tag] = line[sp + 2: line.rfind(b'"')]
            elif line.strip():
                seen += 1
                if seen > skip:
                    process(hdr, line, S, ids)
                    n = S["n"]
                    if n % 100_000 == 0 and time.time() - last > 20:
                        last = time.time(); frac = raw.tell() / total_bytes
                        el = last - t0; print(f"  {n:>12,} games  {frac:6.1%} of file  elapsed {el/60:5.1f} min  ETA {el/max(frac,1e-9)*(1-frac)/60:5.1f} min", flush=True)
                    if n % CHECKPOINT_EVERY == 0:
                        flush_ids()
                        pickle.dump({"fingerprint": fp, "state": S}, open(str(ckpt_path) + ".tmp", "wb")); os.replace(str(ckpt_path) + ".tmp", ckpt_path)
                    if max_games and n >= max_games:
                        partial = True; break
                hdr = {}
        flush_ids()
    finally:
        raw.close()
        if ids_f: ids_f.close()
    S["partial"] = partial; S["fingerprint"] = fp
    return S

def duplicate_check(ids_path):
    a = np.fromfile(ids_path, dtype=np.uint64)
    out = dict(ids_read=int(a.size), zero_ids=int((a == 0).sum()))
    a.sort()
    out["duplicate_ids"] = int((a[1:] == a[:-1]).sum()); out["distinct_ids"] = out["ids_read"] - out["duplicate_ids"]
    return out

def code_fingerprint():
    """Hash of the profiling code and its constants: editing this cell automatically invalidates old checkpoints."""
    consts = repr([THRESH, sorted(WANT), SPEED_KEYS, MOVE_NO.pattern, TC.pattern, EVAL.pattern, VALID_RESULTS, MONTH_DOTTED, BAND_W, BAND_B, BAND_E, CODE_VERSION, MIN_EST_SECONDS, MAX_RATING_GAP])
    try: src = "".join(inspect.getsource(f) for f in (speed_of, est_seconds, to_elo, full_moves, new_state, eval_check, legality_check, process, open_pgn, make_fp, profile, duplicate_check))
    except (OSError, TypeError): src = "source-unavailable"
    return hashlib.sha256((src + consts).encode()).hexdigest()[:16]
CODE_FP = code_fingerprint(); print("code fingerprint:", CODE_FP)

STATE_FILE, CKPT, IDS = OUT / "profile_state.pkl", OUT / "checkpoint.pkl", OUT / "game_ids.u64"
fp_now = make_fp(RAW, MAX_GAMES)
done_before = STATE_FILE.exists() and pickle.load(open(STATE_FILE, "rb")).get("fingerprint") == fp_now
if done_before:
    print("Profile already finished for these settings and this code - loading it (delete profile_state.pkl to recompute).")
    S = pickle.load(open(STATE_FILE, "rb"))
else:
    t_start = time.time()
    S = profile(RAW, MAX_GAMES, CKPT, IDS)
    if CHECK_DUPLICATE_IDS:
        print("Checking the game IDs for duplicates (sorting about 0.8 GB) ...")
        S["dup"] = duplicate_check(IDS); print("  ", S["dup"])
    pickle.dump(S, open(STATE_FILE, "wb"))
    for p in (CKPT, IDS):
        if p.exists(): p.unlink()
    print(f"Done: {S['n']:,} games in {(time.time()-t_start)/60:.1f} min" + ("  (PARTIAL: MAX_GAMES reached)" if S["partial"] else ""))

## 4. Save all tables (CSV), the data-quality checklist and the results hash
Only the tables listed in `HASHED` enter `results_sha256` (a leftover file from an older run cannot change it). CSVs use `\n` line endings and fixed number formats, so Windows, Mac and Linux give the same hash.

In [ ]:
dec = lambda b: b.decode("utf-8", "replace") if isinstance(b, bytes) else str(b)
HASHED = []                                               # exactly these tables go into results_sha256 (no stale files can sneak in)
def save_csv(name, df):
    df.to_csv(TABLES / f"{name}.csv", index=False, lineterminator="\n", float_format="%.10g"); HASHED.append(name)
def counter_df(c, col, top=None):
    items = c.most_common(top) if top else sorted(c.items(), key=lambda kv: kv[0])
    df = pd.DataFrame([(dec(k), v) for k, v in items], columns=[col, "games"]); df["share"] = df.games / S["n"]; return df

save_csv("results", counter_df(S["result"], "result")); save_csv("terminations", counter_df(S["term"], "termination"))
save_csv("speed", counter_df(S["speed"], "speed")); save_csv("time_controls_top40", counter_df(S["tc"], "time_control", 40))
save_csv("titles", counter_df(S["titles"], "title")); save_csv("eco_top30", counter_df(S["eco"], "eco", 30))
save_csv("games_per_day", counter_df(S["day"], "date"))
tf = pd.DataFrame([(dec(k), v) for k, v in S["tags"].most_common()], columns=["tag", "games_with_tag"])
tf["share"] = tf.games_with_tag / min(S["n"], TAG_SAMPLE_GAMES); save_csv("tag_fill_rates", tf)
elo = pd.DataFrame({"elo": sorted(set(S["w"]) | set(S["b"]) | set(S["mn"]))})
for k, col in (("w", "white"), ("b", "black"), ("mn", "min_of_both")): elo[col] = elo.elo.map(S[k]).fillna(0).astype(int)
save_csv("elo_exact", elo)
save_csv("rating_gap_exact", pd.DataFrame(sorted(S["gap"].items()), columns=["gap", "games"]))
save_csv("full_moves", pd.DataFrame(sorted(S["moves"].items()), columns=["full_moves", "games"]))
save_csv("full_moves_by_speed", pd.DataFrame([(a, m, v) for (a, m), v in sorted(S["moves_speed"].items())], columns=["speed", "full_moves", "games"]))
for key, name in (("eval_speed", "eval_by_speed"), ("clk_speed", "clock_by_speed")):
    save_csv(name, pd.DataFrame([(a, h, v) for (a, h), v in sorted(S[key].items())], columns=["speed", "has", "games"]))
save_csv("bots_by_speed", pd.DataFrame([(a, h, v) for (a, h), v in sorted(S["bot"].items())], columns=["speed", "has_bot", "games"]))
save_csv("eval_by_min_elo_band", pd.DataFrame([(a, h, v) for (a, h), v in sorted(S["eval_band"].items())], columns=["min_elo_band", "has_eval", "games"]))
save_csv("cohort_counts", pd.DataFrame([(*k, v) for k, v in sorted(S["cohort"].items())],
         columns=["tier_idx", "gap_ok", "rapid_plus", "normal", "result_ok", "no_bot", "has_eval", "games"]))
# data-quality tables
save_csv("eval_hist", pd.DataFrame(sorted(S["eval_hist"].items()), columns=["eval_cp_from", "evals"]))
save_csv("eval_final_vs_result", pd.DataFrame([(b_, dec(r_), v) for (b_, r_), v in sorted(S["eval_final"].items())], columns=["final_eval_band", "result", "games"]))
save_csv("eval_checks", pd.DataFrame(sorted(S["eval_flags"].items()), columns=["check", "count"]))
save_csv("legality_sample", pd.DataFrame(sorted(S["legal"].items()), columns=["check", "count"]))
save_csv("quality_counts", pd.DataFrame(sorted(S["quality"].items()), columns=["check", "count"]))
save_csv("duplicate_ids", pd.DataFrame(sorted((S["dup"] or {}).items()), columns=["check", "count"]))

# the checklist: one row per check, with a verdict
Nn, Qc, Lg, Fl, Dp = S["n"], S["quality"], S["legal"], S["eval_flags"], (S["dup"] or {})
fin = S["eval_final"]
def agree(band, win):
    tot = sum(v for (b_, r_), v in fin.items() if b_ == band)
    return f"{fin.get((band, win), 0) / tot:.1%} of {tot:,} games" if tot else "no data"
zero = lambda x: "PASS" if x == 0 else "REVIEW"
cnt = lambda x: f"{x:,}" + (f" ({x / Nn:.4%})" if Nn else "")
rows = [
    ("File is complete (size equals the size the server announced)", f"{manifest['bytes']:,} bytes", "PASS" if manifest["bytes"] == expected else "REVIEW"),
    ("SHA-256 equals Lichess's published checksum", {None: "checksum list not available", True: "match", False: "MISMATCH"}[manifest["matches_official"]], {None: "INFO", True: "PASS", False: "REVIEW"}[manifest["matches_official"]]),
    ("Games read", f"{Nn:,}" + (" (PARTIAL test run)" if S["partial"] else " (whole file)"), "INFO"),
    ("Duplicate game IDs (all games)", cnt(Dp["duplicate_ids"]) if Dp else "not run", zero(Dp["duplicate_ids"]) if Dp else "NOT RUN"),
    ("Game IDs that could not be read (bad Site tag)", cnt(Qc["bad_site_tag"]), zero(Qc["bad_site_tag"])),
    ("Missing or invalid Elo (all games)", cnt(Qc["missing_or_invalid_elo"]), zero(Qc["missing_or_invalid_elo"])),
    ("UTCDate outside the month (all games)", cnt(Qc["utcdate_outside_month"]), zero(Qc["utcdate_outside_month"])),
    ("Result at the end of the move text differs from the Result header (all games)", cnt(Qc["movetext_result_differs_from_header"]), zero(Qc["movetext_result_differs_from_header"])),
    ("Games without any move", cnt(Qc["no_moves"]), "INFO"),
    ("Unfinished games (Result *)", cnt(Qc["unfinished_result_star"]), "INFO"),
    ("Games without a clock time control (correspondence)", cnt(Qc["no_clock_time_control"]), "INFO"),
    ("Analysed games with fewer evals than plies - 1 (all analysed games)", cnt(Qc["eval_games_with_missing_evals"]), "INFO"),
    (f"Legality replay: games replayed (every {LEGALITY_EVERY:,}th game)", f"{Lg['games_replayed']:,}", "INFO"),
    ("Legality replay: games with an illegal or unparsable move", cnt(Lg["illegal_or_unparsable_moves"]) if Lg["games_replayed"] else "not run", zero(Lg["illegal_or_unparsable_moves"]) if Lg["games_replayed"] else "NOT RUN"),
    ("Legality replay: parser exceptions / no game found", f"{Lg['parser_exception'] + Lg['no_game_found']:,}", zero(Lg["parser_exception"] + Lg["no_game_found"])),
    ("Legality replay: our game-length estimate differs from the replayed length", f"{Lg['length_estimate_differs']:,}", zero(Lg["length_estimate_differs"])),
    (f"Eval values checked (every {EVAL_EVERY}th analysed game)", f"{Fl['values']:,} values in {Fl['games_checked']:,} games", "INFO"),
    ("Eval tokens that are not numbers or mates", f"{Fl['unparsable']:,}", zero(Fl["unparsable"])),
    ("Eval values beyond +-30 pawns", f"{Fl['abs_over_30_pawns']:,}", "INFO"),
    ("Last eval >= +3.00 and White won", agree(BAND_W, b"1-0"), "INFO"),
    ("Last eval <= -3.00 and Black won", agree(BAND_B, b"0-1"), "INFO"),
]
checklist = pd.DataFrame(rows, columns=["check", "result", "verdict"]); save_csv("data_quality_checklist", checklist)

overview = dict(month=MONTH, games=S["n"], partial=S["partial"], tag_sample_games=min(S["n"], TAG_SAMPLE_GAMES), data_quality=dict(S["quality"]),
                duplicates=S["dup"], legality_sample=dict(S["legal"]), eval_checks=dict(S["eval_flags"]), eval_games=S["eval_games"])
with open(TABLES / "overview.json", "w", encoding="utf-8", newline="\n") as fh: json.dump(overview, fh, indent=2, sort_keys=True)

h = hashlib.sha256()
for name in sorted(HASHED):
    h.update(f"{name}.csv".encode()); h.update((TABLES / f"{name}.csv").read_bytes())
h.update(b"overview.json"); h.update((TABLES / "overview.json").read_bytes())
results_sha = h.hexdigest()
pm = dict(step="step0_profile", month=MONTH, file_sha256=manifest["sha256"], games=S["n"], partial=S["partial"], code_version=CODE_VERSION, code_fingerprint=CODE_FP,
          results_sha256=results_sha, hashed_tables=sorted(HASHED) + ["overview.json"],
          timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
          python=sys.version.split()[0], platform=platform.platform(), packages=pkg_versions(),
          parameters=dict(MAX_GAMES=MAX_GAMES, TAG_SAMPLE_GAMES=TAG_SAMPLE_GAMES, CHECK_DUPLICATE_IDS=CHECK_DUPLICATE_IDS, LEGALITY_EVERY=LEGALITY_EVERY, EVAL_EVERY=EVAL_EVERY))
(OUT / "profile_manifest.json").write_text(json.dumps(pm, indent=2))
print("tables saved in", TABLES, "\nresults_sha256 =", results_sha)

## 5. Key facts and the data-quality checklist
The numbers you will quote in the thesis. Percentiles are exact (computed from per-rating counts) over all player-rating observations (White and Black).
`PASS` = zero problems found, `REVIEW` = look at it, `INFO` = for your description of the data, `NOT RUN` = the check was switched off.

In [ ]:
T = lambda n: pd.read_csv(TABLES / f"{n}.csv")
N = S["n"]
speed, ev, clk, bots = T("speed"), T("eval_by_speed"), T("clock_by_speed"), T("bots_by_speed")
elo = T("elo_exact"); allr = (elo.white + elo.black).values; cum = np.cumsum(allr) / allr.sum()
pct = {p: int(elo.elo.values[np.searchsorted(cum, p / 100)]) for p in (50, 75, 90, 95, 99, 99.9)}
mnc = np.cumsum(elo.min_of_both.values[::-1])[::-1]
share_min = lambda t: elo.min_of_both[elo.elo >= t].sum() / elo.min_of_both.sum()
rapid_plus = speed[speed.speed.isin(["Rapid", "Classical"])].games.sum()
facts = {
    "games in file": f"{N:,}", "compressed size": f"{manifest['bytes']/1e9:.2f} GB",
    "rapid or classical (Event tag)": f"{rapid_plus/N:.2%}",
    "games with Lichess evals": f"{ev[ev.has == 1].games.sum()/N:.2%}",
    "games with clock times": f"{clk[clk.has == 1].games.sum()/N:.2%}",
    "games with a bot account": f"{bots[bots.has_bot == 1].games.sum()/N:.2%}",
    "median full moves": int(T('full_moves').pipe(lambda d: d.full_moves.values[np.searchsorted(np.cumsum(d.games) / d.games.sum(), 0.5)])),
    "rating percentiles (all players)": ", ".join(f"p{p}={v}" for p, v in pct.items()),
    "games with BOTH players >= 2400 / 2500 / 2600": " / ".join(f"{share_min(t):.2%}" for t in (2400, 2500, 2600)),
}
display(pd.Series(facts, name="value").to_frame()); print("data quality:", dict(S["quality"]))
print("\ndata-quality checklist:")
display(pd.read_csv(TABLES / "data_quality_checklist.csv"))

## 6. Plots
Every figure is saved in `outputs/step0_dataset/<run>/figures/`. Figure 9 is the evaluation plausibility check.

In [ ]:
COL = {"main": "#595959", "acc": "#E24A0A", "teal": "#2A9D8F", "light": "#CFCDC9"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10, "axes.ymargin": 0.1})
def save(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=200, bbox_inches="tight"); plt.show()
rebin = lambda s, w: s.groupby(s.index // w * w).sum()

# 1. when / what kind of games
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
d = T("games_per_day"); ax[0].plot(d.date.str[-2:], d.games, color=COL["main"], marker="o", ms=3); ax[0].set_title("Games per day of the month"); ax[0].set_xlabel("day"); ax[0].tick_params(axis="x", labelsize=7)
order = [s for s in ["UltraBullet", "Bullet", "Blitz", "Rapid", "Classical", "Correspondence", "Other"] if s in set(speed.speed)]
sp = speed.set_index("speed").reindex(order); ax[1].bar(sp.index, sp.share * 100, color=COL["main"]); ax[1].set_title("Speed category (Event tag), % of games")
for i, v in enumerate(sp.share * 100): ax[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax[1].tick_params(axis="x", rotation=30)
t = T("time_controls_top40").head(15); ax[2].barh(t.time_control[::-1], t.share[::-1] * 100, color=COL["main"]); ax[2].set_title("Top 15 time controls (base+increment, s), %")
save("01_days_speed_timecontrol")

# 2. results, terminations
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
r = T("results"); ax[0].bar(r.result, r.share * 100, color=COL["main"]); ax[0].set_title("Results, % of games")
for i, v in enumerate(r.share * 100): ax[0].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
tm = T("terminations"); ax[1].barh(tm.termination[::-1], tm.share[::-1] * 100, color=COL["main"]); ax[1].set_title("Termination, % of games")
for i, v in enumerate(tm.share[::-1] * 100): ax[1].text(v, i, f" {v:.2f}%", va="center", fontsize=8)
save("02_results_terminations")

# 3. ratings
elo = T("elo_exact").set_index("elo")
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
b = rebin(elo.white, 50); w = rebin(elo.black, 50)
ax[0, 0].bar(b.index, b.values, width=48, alpha=0.6, color=COL["main"], label="White"); ax[0, 0].bar(w.index, w.values, width=48, alpha=0.6, color=COL["acc"], label="Black")
ax[0, 0].set_title("Player ratings (50-point bins)"); ax[0, 0].legend()
m = rebin(elo.min_of_both, 50); ax[0, 1].bar(m.index, m.values, width=48, color=COL["main"]); ax[0, 1].set_yscale("log")
ax[0, 1].set_title("Rating of the WEAKER player of each game (log y)")
xs = np.arange(1500, 3000, 25); tot = elo.min_of_both.sum()
surv = [elo.min_of_both[elo.index >= x].sum() / tot for x in xs]
ax[1, 0].plot(xs, surv, color=COL["main"]); ax[1, 0].set_yscale("log"); ax[1, 0].set_title("Share of games where BOTH players are rated at least X (log y)")
for tt in (2200, 2400, 2500, 2600):
    v = elo.min_of_both[elo.index >= tt].sum() / tot; ax[1, 0].plot(tt, v, "o", color=COL["acc"]); ax[1, 0].annotate(f"{tt}: {v:.2%}", (tt, v), textcoords="offset points", xytext=(6, 6), fontsize=8)
g = T("rating_gap_exact").set_index("gap").games; g = rebin(g.clip(upper=None), 25); g = g[g.index <= 1000]
ax[1, 1].bar(g.index, g.values, width=24, color=COL["main"]); ax[1, 1].set_yscale("log"); ax[1, 1].axvline(MAX_RATING_GAP, color=COL["acc"], ls="--", label=f"gap = {MAX_RATING_GAP}"); ax[1, 1].legend()
ax[1, 1].set_title("Rating gap between the players (25-point bins, log y)")
save("03_ratings")

# 4. game length
fm = T("full_moves"); fms = T("full_moves_by_speed")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
ax[0].bar(fm.full_moves, fm.games, width=1, color=COL["main"]); ax[0].set_title("Game length (full moves, capped at 300)"); ax[0].set_xlim(0, 150)
for s_ in [x for x in ["Bullet", "Blitz", "Rapid", "Classical"] if x in set(fms.speed)]:
    q = fms[fms.speed == s_].set_index("full_moves").games; ax[1].plot(q.index, q.values / q.sum(), label=s_)
ax[1].set_xlim(0, 120); ax[1].legend(); ax[1].set_title("Game length by speed (share of that speed)")
save("04_game_length")

# 5. titles and bots
tt = T("titles"); bt = T("bots_by_speed").pivot_table(index="speed", columns="has_bot", values="games", fill_value=0).reindex(order)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].bar(tt.title, tt.games, color=COL["main"]); ax[0].set_yscale("log"); ax[0].set_title("Player titles (counts, log y)")
share_bot = (bt.get(1, 0) / bt.sum(axis=1) * 100); ax[1].bar(share_bot.index, share_bot.values, color=COL["acc"]); ax[1].set_title("Games with a BOT account, % by speed"); ax[1].tick_params(axis="x", rotation=30)
for i, v in enumerate(share_bot.values): ax[1].text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=8)
save("05_titles_bots")

# 6. evals and clocks
evs = T("eval_by_speed").pivot_table(index="speed", columns="has", values="games", fill_value=0).reindex(order)
ck = T("clock_by_speed").pivot_table(index="speed", columns="has", values="games", fill_value=0).reindex(order)
eb = T("eval_by_min_elo_band").pivot_table(index="min_elo_band", columns="has_eval", values="games", fill_value=0)
eb = eb[eb.index >= 1000]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(len(order)); ax[0].bar(x - 0.2, evs.get(1, 0) / evs.sum(axis=1) * 100, 0.4, color=COL["teal"], label="with [%eval]")
ax[0].bar(x + 0.2, ck.get(1, 0) / ck.sum(axis=1) * 100, 0.4, color=COL["main"], label="with [%clk]"); ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=30); ax[0].set_ylim(0, 110); ax[0].legend(loc="upper right", ncol=2, fontsize=8); ax[0].set_title("Share of games with evals / clocks by speed, %")
ax[1].plot(eb.index, eb.get(1, 0) / eb.sum(axis=1) * 100, marker="o", color=COL["teal"]); ax[1].set_title("Share of games with evals by rating of the weaker player, %"); ax[1].set_xlabel("rating band (100 points)")
ax[2].bar(eb.index, eb.sum(axis=1), width=90, color=COL["light"]); ax[2].set_yscale("log"); ax[2].set_title("Games per rating band (log y)")
save("06_evals_clocks")

# 7. openings, tag fill
e = T("eco_top30").head(20); tf = T("tag_fill_rates")
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].barh(e.eco[::-1], e.share[::-1] * 100, color=COL["main"]); ax[0].set_title("Top 20 ECO opening codes, % of games")
ax[1].barh(tf.tag[::-1], tf.share[::-1] * 100, color=COL["main"]); ax[1].set_xlim(0, 105); ax[1].set_title(f"Header tag fill rate (first {min(S['n'], TAG_SAMPLE_GAMES):,} games), %")
save("07_openings_tags")

# 9. are the Lichess evaluations plausible?
eh = T("eval_hist"); ef = T("eval_final_vs_result")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
ax[0].bar(eh.eval_cp_from + 50, eh.evals, width=95, color=COL["main"]); ax[0].set_yscale("log")
ax[0].set_title(f"Eval values in centipawns (White's view; ends hold everything beyond +-10 pawns)\nevery {EVAL_EVERY}th analysed game", fontsize=9)
if len(ef):
    pv = ef.pivot_table(index="final_eval_band", columns="result", values="games", fill_value=0); pv = pv.div(pv.sum(axis=1), axis=0) * 100
    pv.plot(kind="bar", ax=ax[1], rot=0); ax[1].set_title("Game result by the LAST evaluation of the game, % (plausibility check)", fontsize=9)
save("09_eval_plausibility")

# 8. what-if funnel for different rating tiers (same order as the thesis pipeline)
co = T("cohort_counts"); stages = ["both >= tier", f"+ gap <= {MAX_RATING_GAP}", f"+ rapid or slower (>= {MIN_EST_SECONDS} s)", "+ normal termination", "+ valid result", "+ no bots", "+ has evals"]
rows = {}
for ti, tv in enumerate(THRESH):
    c = co[co.tier_idx >= ti]; seq = [c.games.sum()]
    for col in ["gap_ok", "rapid_plus", "normal", "result_ok", "no_bot", "has_eval"]:
        c = c[c[col] == 1]; seq.append(c.games.sum())
    rows[f">= {tv}"] = seq
funnel = pd.DataFrame(rows, index=stages)
fig, ax = plt.subplots(figsize=(11, 5))
for colname in funnel.columns: ax.plot(range(len(stages)), funnel[colname].clip(lower=0.5), marker="o", label=colname)
ax.set_yscale("log"); ax.set_xticks(range(len(stages))); ax.set_xticklabels(stages, rotation=15); ax.legend(title="both players rated", ncol=2)
ax.set_title("What-if: games left after each filter, for different rating tiers (log y)")
save("08_funnel_by_rating_tier"); funnel.to_csv(TABLES / "funnel_by_rating_tier.csv", lineterminator="\n"); funnel

## 7. Reproducibility check and cross-check against the earlier independent run
Run the whole notebook again (or on another computer with the same file): `file_sha256` and `results_sha256` below must be identical.
The cross-check compares key numbers with the independent script run made earlier (`profile_raw_dataset.py` / `count_eligible_elite.py`): different code, same file, so the numbers must match.
Commit `outputs/step0_dataset/<month>/` (manifests, tables, figures) to git, but **not** `data/raw/` (see `.gitignore`).

In [ ]:
pm = json.loads((OUT / "profile_manifest.json").read_text()); dm = json.loads(MAN.read_text())
print("file sha256      :", dm["sha256"]); print("file bytes       :", f"{dm['bytes']:,}"); print("results_sha256   :", pm["results_sha256"])
print("code fingerprint :", pm["code_fingerprint"])
print("games profiled   :", f"{pm['games']:,}", "(PARTIAL)" if pm["partial"] else "(whole file)")
print("hashed tables    :", len(pm["hashed_tables"]), "files")
print("figures          :", *sorted(p.name for p in FIGS.glob("*.png")), sep="\n  ")

if MONTH == "2026-10" and not pm["partial"] and (MAX_RATING_GAP, MIN_EST_SECONDS) == (100, 480):
    co = T("cohort_counts"); c = co[co.tier_idx >= THRESH.index(2600)]
    s1 = int(c.games.sum()); c = c[c.gap_ok == 1]; s2 = int(c.games.sum()); c = c[c.rapid_plus == 1]; s3 = int(c.games.sum())
    c = c[(c.normal == 1) & (c.result_ok == 1) & (c.no_bot == 1)]; s4 = int(c.games.sum())
    got = {"games": S["n"], "games with Lichess evals": int(T("eval_by_speed").query("has == 1").games.sum()),
           "games with clock times": int(T("clock_by_speed").query("has == 1").games.sum()),
           "games with a bot account": int(T("bots_by_speed").query("has_bot == 1").games.sum()),
           "both players >= 2600": s1, "+ rating gap <= 100": s2, "+ rapid or slower": s3, "+ normal, valid result, no bot": s4}
    exp = {"games": 95_395_628, "games with Lichess evals": 9_409_507, "games with clock times": 94_898_369, "games with a bot account": 494_406,
           "both players >= 2600": 534_440, "+ rating gap <= 100": 371_153, "+ rapid or slower": 11_159, "+ normal, valid result, no bot": 175}
    cc = pd.DataFrame({"this notebook": got, "earlier run": exp}); cc["verdict"] = np.where(cc["this notebook"] == cc["earlier run"], "MATCH", "MISMATCH")
    display(cc)
else:
    print("(cross-check against the earlier run is only available for the whole 2026-10 file with gap 100 and rapid-or-slower >= 480 s)")